In [ ]:
# Project settings and paths: config.py is in the project root (this also makes the modules in src/ importable)
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())))
import config

# Standard libraries
import numpy as np
from pathlib import Path
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import tempfile
import os
import subprocess

# Specific libraries
import nibabel as nib
import seaborn as sns

# Import custom modules
import file_io as fio
import dicom_load as dcmload
import registration as reg

In [ ]:
# Define directories
# Target dataset to preprocess: the phantom or the in vivo data
TARGET_DIR = config.INVIVO_DIR           # config.CALIBER_DIR or config.INVIVO_DIR
IS_BRAIN = True                          # True: brain extraction with FSL BET. False: threshold based mask

# How the scans are registered (see REGISTRATION below)
#   "paired":    b=0 scans are translated onto the first b=0, each b=800 follows its b=0, then is scaled onto it
#   "all_to_b0": every scan is translated then scaled onto the first b=0 scan
REGISTRATION = "paired"
REFERENCE = "middle"                     # "paired" only: reference b=0 scan: "first", "middle" (most central position) or an index
B0_TRANSFORM = "rigid"                  # "rigid" (rotation + translation) or "translation" for the first registration step
                                         # ("paired": the b=0 scans, "all_to_b0": every scan, before the scaling step)

SCAN_DIR = TARGET_DIR / "scan_data"
OUTPUT_DIR = TARGET_DIR / "derivatives"
testname = TARGET_DIR.stem

if not TARGET_DIR.is_dir():
    raise NotADirectoryError(f"Expected a directory, got: {TARGET_DIR}")
if not SCAN_DIR.is_dir():
    raise NotADirectoryError(f"{TARGET_DIR.stem} doesn't contain subdirectory named {SCAN_DIR.stem}")

OUTPUT_DIR.mkdir(exist_ok=True)

### LOAD & UNPACK

In [ ]:
# Load in data and save basic features 
dicom_fnames = fio.scrape_files(SCAN_DIR, filetype=["MRe", "dcm"])
dicom_fnames.sort(key = lambda x: dcmload.dicom_scantime(x))
print(f"The full dataset contains {len(dicom_fnames)} DICOM files")

dataset_unpacked = list(map(dcmload.process_dicom, dicom_fnames))
dataset_df = pd.DataFrame(dataset_unpacked)
print("Scans unpacked into pandas DataFrame structure")

In [ ]:
def plot_num_of_repeats(df, col_name):

    num_repeats = df.groupby(col_name).size().rename("count").reset_index()

    fig, ax = plt.subplots(figsize=(16, 9))
    sns.barplot(data=num_repeats, x=col_name, y="count", hue=col_name, palette="husl", ax=ax)

    # Tidy up axes
    ax.set_xticks(np.arange(num_repeats.values.shape[0]), num_repeats[col_name])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center")
    ax.set_yticks(range(0, num_repeats["count"].values.max() + 1))
    ax.set_ylabel("Number of Repeats")
    fig.tight_layout()

    plt.show()

plot_num_of_repeats(dataset_df, "seq_name")

### REGISTRATION
The target scans are registered as noisy scans (there are too few volumes for MP-PCA). No MP-PCA is used.

- `"paired"`: the protocol acquires a b=0 and then a b=800 scan for each direction, so the two are close in time. The b=0 scans
  are registered (rigid or translation, `B0_TRANSFORM`) to the reference b=0 scan (`REFERENCE`; same contrast, so reliable). Each b=800 scan receives the translation of
  its b=0 partner and is then scaling registered onto that b=0 scan, which corrects the diffusion-gradient specific distortion.
- `"all_to_b0"`: every scan is registered (rigid or translation, `B0_TRANSFORM`) and then scaled onto the first b=0 scan.

In [ ]:
# Extract images from dataframe
image_4d = np.stack(dataset_df["image"].values, axis=-1)
aff_0 = dataset_df["affine"].iloc[0]
spacing_0 = dataset_df["spacing"].iloc[0]

# Extract b-values in order to be able to identify b=0 scan to register other scans to
bvals = dataset_df["b_value"].values

# Perform registration:
# function returns a registered 4D image array and also
# a numpy array of indices for which registration failed
if REGISTRATION == "paired":
    # Scans are in acquisition order (sorted by scan time when loaded), each b=800 follows its b=0
    reg_4d, i_drop = reg.register_pairs_to_b0(data_4d=image_4d,
                                              bvals=bvals,
                                              directions=dataset_df["dwi_direction"].values,
                                              spacing=spacing_0,
                                              b0_threshold=5,
                                              b0_transform=B0_TRANSFORM,
                                              reference=REFERENCE)
elif REGISTRATION == "all_to_b0":
    reg_4d, i_drop = reg.register_to_b0(data_4d=image_4d,
                                        bvals=bvals,
                                        spacing=spacing_0,
                                        b0_threshold=5,
                                        first_transform=B0_TRANSFORM)
else:
    raise ValueError(f"Unknown REGISTRATION: {REGISTRATION}")

# Create a new dataframe with only the measurements that were successfully
# registered and append a column for registered scans
if len(i_drop) > 0:
    print(f"Registration failed for {len(i_drop)} scans, these are removed: {list(i_drop)}")
reg_df = dataset_df.drop(index=i_drop).reset_index(drop=True)
reg_df["reg_image"] = list(np.transpose(reg_4d, (3,0,1,2)))

# Every direction needs a diffusion-weighted scan, and there must be a b=0 scan
if not (reg_df["b_value"] < 5).any():
    raise ValueError("No b=0 scan remains after registration")
lost = set(dataset_df["dwi_direction"].unique()) - set(reg_df["dwi_direction"].unique())
if lost:
    raise ValueError(f"No scans remain for direction(s) {lost} after registration")

### CREATE b=0 MASK

In [ ]:
print(f"Subject is a brain: {IS_BRAIN}")

In [ ]:
def run_bet_on_array(vol: np.ndarray, affine: np.ndarray | None = None, frac: float = 0.5):

    with tempfile.TemporaryDirectory() as tmpdir:
        in_path = os.path.join(tmpdir, "input.nii.gz")
        out_base = os.path.join(tmpdir, "brain")

        img = nib.Nifti1Image(vol.astype(np.float32), affine)
        nib.save(img, in_path)

        cmd = ["bet", in_path, out_base, "-f", str(frac)]

        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        except FileNotFoundError:
            raise RuntimeError("FSL 'bet' was not found on the path. Install FSL to extract the brain, "
                               "or supply a brain mask.")
        except subprocess.CalledProcessError as e:
            raise RuntimeError(f"bet failed: {e.stderr}")
        brain = nib.load(out_base + ".nii.gz").get_fdata()

        return brain

b0_threshold = 5
# Mask is created from the registered b=0 scans
b0_scans = np.stack(reg_df[reg_df["b_value"] < b0_threshold]["reg_image"].values, axis=-1)
b0_avg = np.mean(b0_scans, axis=-1)

if IS_BRAIN:
    # If target data is a brain we can run BET2 to extract the brain region
    mask = run_bet_on_array(b0_avg, affine=aff_0, frac=0.73).astype(bool)

else:
    # If a phantom we shall just use ants to create the mask 
    brain = reg.get_mask(b0_avg)
    mask = brain > 0

# Plot the resulting masked data
plotdata = b0_avg
plotdata[mask==0] = 0
zmin = np.percentile(plotdata[mask], 1)
zmax = np.percentile(plotdata[mask], 99)
px.imshow(plotdata, animation_frame=2, zmin=zmin, zmax=zmax, height=800, width=1000, title="Masked b=0 image")

## SAVE OUTPUTS

In [ ]:
# One scan per sequence expected (more would be silently dropped)
if reg_df.groupby("seq_name").size().max() > 1:
    raise ValueError("More than one scan per sequence, repeats would be discarded")
output_df = reg_df.sort_values("seq_name").reset_index(drop=True)   # same order as before
print(output_df.shape)

raw_images = np.stack(output_df["image"].values, axis=-1)
raw_nii = nib.Nifti1Image(raw_images, affine=aff_0)
fio.save_nifti(raw_nii, OUTPUT_DIR, testname + "_" + "RAW.nii.gz")

reg_images = np.stack(output_df["reg_image"].values, axis=-1)
reg_nii = nib.Nifti1Image(reg_images, affine=aff_0)
fio.save_nifti(reg_nii, OUTPUT_DIR, testname + "_" + "RAW_REG.nii.gz")

b0_mask_nii = nib.Nifti1Image(mask.astype(float), aff_0)
fio.save_nifti(b0_mask_nii, OUTPUT_DIR, testname + "_" + "MASK.nii.gz")

param_df = output_df.drop(["image", "reg_image"], axis=1, inplace=False)
param_fname = OUTPUT_DIR / (testname + "_" + "params.h5")
param_df.to_hdf(param_fname, key="df", mode="w", index=False)